In [ ]:
import os
import random

import kagglehub
import numpy as np
import torch
import torch.nn.functional as F
from sklearn import metrics
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [ ]:
# --- Seed Setup ---
seed = 100
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
# --- Metric Functions ---
def sim_auc(similarities, datasets):
    """
    Calculate AUC and FPR95 for multiple OOD datasets against ID dataset.
    """
    if len(similarities) != len(datasets):
        raise ValueError(
            "Number of similarities arrays must match number of dataset names"
        )

    if len(similarities) < 2:
        raise ValueError("At least 2 datasets (ID and OOD) are required")

    similarities = np.array(
        similarities, dtype=object
    )  # Use object dtype for arrays of different lengths
    id_confi = similarities[0]

    auc_scores = []
    fpr_scores = []

    for ood_confi, dataset in zip(similarities[1:], datasets[1:]):
        auroc, fpr_95 = calculate_auc_metrics(id_confi, ood_confi)
        auc_scores.append(auroc)
        fpr_scores.append(fpr_95)
        print(f"Dataset: {dataset:<25} | AUC: {auroc:.4f} | FPR95: {fpr_95:.4f}")

    avg_auc = np.mean(auc_scores)
    avg_fpr = np.mean(fpr_scores)

    print("-" * 60)
    print(f"Average AUC: {avg_auc:.4f} | Average FPR95: {avg_fpr:.4f}")

    return avg_auc, avg_fpr


def sim_ap(similarities, datasets):
    """
    Calculate Average Precision for multiple OOD datasets against ID dataset.
    """
    if len(similarities) != len(datasets):
        raise ValueError(
            "Number of similarities arrays must match number of dataset names"
        )

    if len(similarities) < 2:
        raise ValueError("At least 2 datasets (ID and OOD) are required")

    similarities = np.array(similarities, dtype=object)
    id_confi = similarities[0]

    ap_scores = []

    for ood_confi, dataset in zip(similarities[1:], datasets[1:]):
        aver_p = calculate_average_precision(id_confi, ood_confi)
        ap_scores.append(aver_p)
        print(f"Dataset: {dataset:<25} | AP: {aver_p:.4f}")

    avg_ap = np.mean(ap_scores)
    print("-" * 40)
    print(f"Average AP: {avg_ap:.4f}")

    return avg_ap


def calculate_auc_metrics(id_conf, ood_conf):
    """
    Calculate AUROC and FPR at 95% TPR for binary classification.
    """
    # Combine predictions and create labels
    all_conf = np.concatenate([id_conf, ood_conf])
    # ID samples are positive (1), OOD samples are negative (0)
    labels = np.concatenate([np.ones(len(id_conf)), np.zeros(len(ood_conf))])

    # Calculate ROC curve
    fpr, tpr, _ = metrics.roc_curve(labels, all_conf)

    # Calculate AUROC
    auroc = metrics.auc(fpr, tpr)

    # Calculate FPR at 95% TPR
    tpr_threshold = 0.95
    valid_indices = tpr >= tpr_threshold
    if np.any(valid_indices):
        fpr_at_95 = fpr[np.argmax(valid_indices)]
    else:
        fpr_at_95 = fpr[-1]
        # print(f"Warning: 95% TPR not achievable. Max TPR: {tpr[-1]:.3f}")

    return auroc, fpr_at_95


def calculate_average_precision(id_predictions, ood_predictions):
    # Combine predictions and create labels
    all_predictions = np.concatenate([id_predictions, ood_predictions])
    # ID samples are positive (1), OOD samples are negative (0)
    labels = np.concatenate(
        [np.ones(len(id_predictions)), np.zeros(len(ood_predictions))]
    )

    # Calculate Average Precision
    average_precision = metrics.average_precision_score(labels, all_predictions)

    return average_precision


# --- Model Definitions ---
DEFAULT_MEAN = (0.485, 0.456, 0.406)
DEFAULT_STD = (0.229, 0.224, 0.225)


class RIGID_Detector:
    def __init__(self, lamb=0.05, percentile=5):
        self.lamb = lamb
        self.model = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14").cuda()
        self.model.eval()

    @torch.no_grad()
    def calculate_sim(self, data):
        features = self.model(data)
        noise = torch.randn_like(data).to(data.device)
        trans_data = data + noise * self.lamb
        trans_features = self.model(trans_data)
        sim_feat = F.cosine_similarity(features, trans_features, dim=-1)
        return sim_feat

    @torch.no_grad()
    def detect(self, data):
        sim = self.calculate_sim(data)
        return sim


transform_RIGID = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=DEFAULT_MEAN, std=DEFAULT_STD),
    ]
)

# --- Main Execution ---

# 1. Download Dataset
print("Downloading/Loading dataset from Kaggle...")
path = kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces")
print("Path to dataset files:", path)

base_path = os.path.join(path, "real_vs_fake", "real-vs-fake")
test_dir = os.path.join(base_path, "test")

if not os.path.exists(test_dir):
    print(f"Warning: {test_dir} not found. checking {path}")
    test_dir = path

noise_intensity = 0.05
batch_size = 64

rigid_detector = RIGID_Detector(lamb=noise_intensity)

with torch.no_grad():
    dataset_folder = datasets.ImageFolder(root=test_dir, transform=transform_RIGID)

    class_to_idx = dataset_folder.class_to_idx
    print(f"Found classes: {class_to_idx}")

    real_idx = class_to_idx["real"]
    fake_idx = class_to_idx["fake"]

    data_loader = DataLoader(
        dataset_folder, batch_size=batch_size, shuffle=True, num_workers=2
    )

    # Use lists to accumulate tensors
    real_scores_acc = []
    fake_scores_acc = []

    total_images_processed = 0

    # Tracking for reporting
    report_interval = 2000
    next_report_threshold = report_interval

    print(f"Running inference... Reporting every {report_interval} images.")

    for i, (samples, labels) in enumerate(data_loader):
        samples = samples.cuda()

        # Calculate scores
        sim = rigid_detector.calculate_sim(samples)

        # Split scores based on labels (Real vs Fake)
        sim_cpu = sim.cpu()

        real_mask = labels == real_idx
        fake_mask = labels == fake_idx

        if real_mask.any():
            real_scores_acc.append(sim_cpu[real_mask])
        if fake_mask.any():
            fake_scores_acc.append(sim_cpu[fake_mask])

        total_images_processed += len(samples)

        # --- INTERIM REPORTING LOGIC ---
        if total_images_processed >= next_report_threshold:
            print(
                f"\n\n>>> Interim Report: Processed {total_images_processed} images <<<"
            )

            # Create temporary numpy arrays for calculation
            # We need both classes to calculate ROC/AUC
            if len(real_scores_acc) > 0 and len(fake_scores_acc) > 0:
                curr_real = torch.cat(real_scores_acc).numpy()
                curr_fake = torch.cat(fake_scores_acc).numpy()

                print(
                    f"Current Stats: Real Samples: {len(curr_real)}, Fake Samples: {len(curr_fake)}"
                )

                curr_datasets = [curr_real, curr_fake]
                curr_names = ["Real Faces", "Fake Faces"]

                print("--- AUC ---")
                sim_auc(curr_datasets, curr_names)
                print("--- AP ---")
                sim_ap(curr_datasets, curr_names)
            else:
                print(
                    "Skipping report: Need at least one Real and one Fake image processed."
                )

            # Update threshold
            next_report_threshold += report_interval

    # --- FINAL REPORT ---
    if len(real_scores_acc) > 0:
        real_scores_final = torch.cat(real_scores_acc).numpy()
    else:
        real_scores_final = np.array([])

    if len(fake_scores_acc) > 0:
        fake_scores_final = torch.cat(fake_scores_acc).numpy()
    else:
        fake_scores_final = np.array([])

    print(f"\n\n========================================")
    print(f"Final Processing Complete.")
    print(f"Real Images (ID): {len(real_scores_final)}")
    print(f"Fake Images (OOD): {len(fake_scores_final)}")

    sim_datasets = [real_scores_final, fake_scores_final]
    dataset_names = ["Real Faces", "Fake Faces"]

    print("\nFinal Detection Results AUC:")
    sim_auc(sim_datasets, dataset_names)

    print("\nFinal Detection Results AP:")
    sim_ap(sim_datasets, dataset_names)

/home/tiberiu/Documents/Disertatie/RIGID/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloading/Loading dataset from Kaggle...
Path to dataset files: /home/tiberiu/.cache/kagglehub/datasets/xhlulu/140k-real-and-fake-faces/versions/2


Using cache found in /home/tiberiu/.cache/torch/hub/facebookresearch_dinov2_main
/home/tiberiu/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/home/tiberiu/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/home/tiberiu/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


Found classes: {'fake': 0, 'real': 1}
Running inference... Reporting every 2000 images.


>>> Interim Report: Processed 2048 images <<<
Current Stats: Real Samples: 1046, Fake Samples: 1002
--- AUC ---
Dataset: Fake Faces                | AUC: 0.9110 | FPR95: 0.4621
------------------------------------------------------------
Average AUC: 0.9110 | Average FPR95: 0.4621
--- AP ---
Dataset: Fake Faces                | AP: 0.9237
----------------------------------------
Average AP: 0.9237


>>> Interim Report: Processed 4032 images <<<
Current Stats: Real Samples: 2050, Fake Samples: 1982
--- AUC ---
Dataset: Fake Faces                | AUC: 0.9109 | FPR95: 0.4400
------------------------------------------------------------
Average AUC: 0.9109 | Average FPR95: 0.4400
--- AP ---
Dataset: Fake Faces                | AP: 0.9215
----------------------------------------
Average AP: 0.9215


>>> Interim Report: Processed 6016 images <<<
Current Stats: Real Samples: 3065, Fake Samples: 2951
--- 